# Stratified Analysis

Stratified analysis creates **separate control charts for each factor level**. This is powerful when:

- Each stream (machine, lane, operator) has its own behavior
- You want to detect changes within individual streams
- Comparing streams directly would mask within-stream signals

## What You'll Learn

1. Create stratified IMR charts for multiple streams
2. Understand when to use stratified vs. combined analysis
3. Navigate between individual stream charts
4. Detect signals within each stratum

## Setup

In [ ]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame

## Create Multi-Stream Data

Simulate a filling line with 4 lanes, each with:
- Different baseline performance
- Different variation levels
- Lane C has a special cause event at time 15

In [ ]:
np.random.seed(42)

lanes = ['Lane_A', 'Lane_B', 'Lane_C', 'Lane_D']
n_times = 20

# Each lane has different characteristics
lane_config = {
    'Lane_A': {'mean': 100, 'std': 1.0},
    'Lane_B': {'mean': 101, 'std': 1.5},
    'Lane_C': {'mean': 99, 'std': 1.2},
    'Lane_D': {'mean': 100.5, 'std': 0.8}
}

data = []
for t in range(n_times):
    for lane in lanes:
        config = lane_config[lane]
        
        # Special cause: Lane C at time 15
        special = 6 if (lane == 'Lane_C' and t == 14) else 0
        
        value = config['mean'] + special + np.random.normal(0, config['std'])
        data.append({
            'batch': t + 1,
            'lane': lane,
            'fillweight': round(value, 2)
        })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} observations")
print(f"Structure: {len(lanes)} lanes x {n_times} batches")
df.head(8)

## Formulate the Study

In [ ]:
pdf = ProcessDataFrame(df)

study = pdf.formulate(
    response=pdf.columns.fillweight,
    factors=[pdf.columns.lane],
    time=pdf.columns.batch
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")

## Combined vs. Stratified Analysis

With factors and time, you have two options:

### Combined Analysis (Xbar-S)
- Compares factor levels against each other
- Uses pooled within-group variance
- Good for detecting **between-group differences**

### Stratified Analysis (IMR per level)
- Each factor level gets its own chart
- Uses within-level variance for each
- Good for detecting **within-group changes over time**

## Create Stratified IMR Charts

In [ ]:
# Analyze with IMR - creates one chart per lane
result = study.analyze(chart='Imr')

print(f"Charts created: {result.all_charts}")
print(f"\nStratified charts: {result.list_strata()}")

## View Individual Stream Charts

In [ ]:
# Get data for Lane A
lane_a_data = result.get_stratified_chart('Lane_A')
print("Lane A Chart Data:")
lane_a_data.head(10)

In [ ]:
# Each lane has its own control limits
for lane in lanes:
    stats = result.get_statistics(f'Imr_{lane}')
    print(f"{lane}: CL={stats['center']:.2f}, UCL={stats['ucl']:.2f}, LCL={stats['lcl']:.2f}")

## Visualize All Lanes Together

Use faceting to see all lanes in one view:

In [ ]:
fig = result.plot(
    facet=True,
    ncols=2,
    show_zones=True,
    show_signals=True
)
fig.show()

## View Individual Lane

In [ ]:
# Focus on Lane C (has the special cause)
fig = result.plot(
    chart='Imr_Lane_C',
    show_zones=True,
    show_signals=True,
    show_rules=True,
    show_stats=True
)
fig.show()

## Detect Signals Across All Lanes

In [ ]:
# Detect signals on all stratified charts
for lane in lanes:
    chart_name = f'Imr_{lane}'
    signals = result.detect_signals(chart=chart_name, rules='extended')
    
    if signals.has_signals:
        print(f"{lane}: {signals.count} signal(s)")
        display(signals.violations)
    else:
        print(f"{lane}: No signals")

## Iterate Through Charts

Use the iterator for programmatic access:

In [ ]:
# Process all charts
for name, data, stats in result.iter_charts():
    print(f"{name}: n={len(data)}, center={stats['center']:.2f}")

## Compare with Combined Analysis

Let's see what the Xbar-S analysis shows:

In [ ]:
# Combined Xbar analysis
result_xbar = study.analyze(chart='Xbar')

fig = result_xbar.plot(
    chart='Xbar',
    show_zones=True,
    show_signals=True,
    title='Combined Xbar Chart'
)
fig.show()

### Key Difference

Notice how the **combined Xbar chart** may show different signals than the **stratified IMR charts**:

- Xbar uses pooled variance across lanes
- Lane-specific variation differences are averaged out
- A signal in one lane might be masked

The **stratified approach** uses each lane's own variation, making it more sensitive to within-lane changes.

## When to Use Each Approach

### Use Combined (Xbar-S) When:
- Comparing lanes/machines/operators to each other
- Looking for systematic differences between groups
- Setting up initial process capability

### Use Stratified (IMR) When:
- Monitoring individual streams over time
- Each stream has different inherent variation
- Want maximum sensitivity to within-stream changes
- Historical data shows streams behave differently

## Summary

In this tutorial, you learned:

- Stratified IMR creates one chart per factor level
- Each stratum has its own control limits
- Use faceted plots to view all strata together
- Stratified analysis is more sensitive to within-stream changes
- Choose stratified vs. combined based on your question

## Next Steps

- {doc}`signal-detection` - All Western Electric rules
- {doc}`../user-guide/chart-types` - When to use each chart type
- {doc}`../user-guide/plotting` - Advanced visualization options